[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/datacrunch-2/quickstarters/quickstarter/quickstarter.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/datacrunch-2/assets/banner.webp)

# DataCrunch 2

## Challenge Overview

Datacrunch uses the quantitative research of the CrunchDAO to manage its systematic market-neutral portfolio. DataCrunch built a dataset covering thousands of publicly traded U.S companies.

The long-term strategic goal of the fund is capital appreciation by capturing idiosyncratic return at low volatility.

In order to achieve this goal, Datacrunch needs the community to assess the relative performance of all assets in a subset of the [Russell 3000](https://www.investopedia.com/terms/r/russell_3000.asp) universe. In other words, DataCrunch is expecting your model to maximise the correlation to the constituent of its investment universe.

# Setup

The first steps to get started are:
1. Get the setup command
2. Execute it in the cell below

### >> https://hub.crunchdao.com/competitions/datacrunch-2/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)

In [9]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook datacrunch-2 5HKWCRt304kaRaPvL65d1ZoN --size small

crunch-cli, version 11.3.0

---
Your token seems to have expired or is invalid.

Please follow this link to copy and paste your new setup command:
https://hub.crunchdao.com/competitions/datacrunch-2/submit

If you think that is an error, please contact an administrator.


# Your model

## Setup

In [10]:
# Imports
import os
import typing

# Specify the library version with the `==` operator.
import joblib # == 1.3.2
import pandas as pd # == 2.1.0
import numpy as np # == 1.24.3

# Import sklearn linear model
import sklearn # == 1.1.3
from sklearn.linear_model import LinearRegression

In [11]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>

cli version: 11.3.0
available ram: 12.67 gb
available cpu: 2 core
----


## Understanding the Data

Each row of the dataset describes a stock at a certain date.

In [12]:
# Load the data simply

X_train, y_train, X_test = crunch_tools.load_data()

.crunchdao/project.json: not found, are you in the project directory?
.crunchdao/project.json: make sure to `cd <competition>` first


Abort: 

### Understanding `X_train`

**Columns:**
- `moon`: A sequentially increasing integer representing a date. Time between subsequent dates is constant, denoting a weekly fixed frequency at which the data is sampled.
- `id`: A unique identifier representing a stock at a given `moon`. Note that the same asset has a different `id` in different `moon`.
- `(Feature_1, …, Feature_n)`: Anonymised features that describe the state of assets on a given date. They are several ways of assessing the relative performance of each stock on a given `moon`.

**Note:**
- All features have the string `Feature_` in their name.

In [ ]:
X_train

In [ ]:
# 计算特征相关系数矩阵
feature_cols = [c for c in X_train.columns if 'Feature' in c]

# 提取特征数据
X_features = X_train[feature_cols]

# 计算相关系数矩阵
corr_matrix = X_features.corr()

print(f"特征数量: {len(feature_cols)}")
print(f"相关系数矩阵形状: {corr_matrix.shape}")

# 找出相同 prefix 的特征组
import re
prefixes = set()
for col in feature_cols:
    match = re.match(r'^([^_]+)_Feature', col)
    if match:
        prefixes.add(match.group(1))
print(f"\n特征前缀: {sorted(prefixes)}")

# 计算各 prefix 组内的平均相关系数
print("\n=== 各特征组内平均相关系数 ===")
for prefix in sorted(prefixes):
    group_cols = [c for c in feature_cols if c.startswith(f'{prefix}_Feature')]
    if len(group_cols) > 1:
        group_corr = corr_matrix.loc[group_cols, group_cols]
        # 获取上三角矩阵（不含对角线）的平均值
        mask = np.triu(np.ones_like(group_corr, dtype=bool), k=1)
        avg_corr = group_corr.where(mask).stack().mean()
        print(f"{prefix}: {len(group_cols)} 个特征, 组内平均相关系数 = {avg_corr:.4f}")

# 显示完整相关系数矩阵（前20个特征）
print("\n=== 相关系数矩阵（前20个特征）===")
print(corr_matrix.iloc[:20, :20].round(3))

### Understanding `y_train`

**Columns:**
- `moon`: Same as in `X_train`.
- `id`: Same as in `X_train`.
- `target`: the target that may help you build your models which is based on 28 days (4 moons) compounded returns.

In [ ]:
y_train

### Understanding `X_test`

`X_test` have the same structure as `X_train` but comprises only a few moons, the ones you must predict.

These files are used to simulate the submission process locally via `crunch_tools.test()`. <br />
The aim is to help participants debug their code and have successful submissions. <br />
A successful local test usually means no errors during execution on the submission platform.

In [ ]:
X_test

## Strategy Implementation

### Utilities

Function used in both `train()` and `infer()`.

In [ ]:
def get_model_path(
    model_directory_path: str,
):
    return os.path.join(
        model_directory_path,
        f"model.joblib"
    )

get_model_path("resources")

In [ ]:
def get_feature_columns(
    X: pd.DataFrame,
):
    return [
        column
        for column in X.columns
        if column.startswith("Feature_")
    ]

get_feature_columns(X_train)[:10]

### The `train()` Function

In this function, you build and train your model for making inferences on the test data. Your model must be stored in the `model_directory_path`.

This function will be called in a frequency that is defined by your `train frequency` parameter that you will define when deploying your model on the Crunch platform.

In [ ]:
# Uncomment what you need!
def train(
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    model_directory_path: str,
    # loop_moon: int,
    # embargo: int,
) -> None:
    """
    Do your model training here.
    At each retrain this function will have to save an updated version of the model under the model_directory_path, as in the example below.
    Note: You can use other serialization methods than joblib.dump(), as long as it matches what reads the model in infer().

    Args:
        X_train, y_train: the data to train the model.
        model_directory_path: the path to save your updated model
        loop_moon: the moon currently being processed
        embargo: data embrago

    Returns:
        None
    """

    model = LinearRegression()

    feature_columns = get_feature_columns(X_train)
    model.fit(X_train[feature_columns], y_train["target"])

    model_path = get_model_path(model_directory_path)
    joblib.dump(model, model_path)

### The `infer()` Function

In the inference function, your trained model (if any) is loaded and used to make predictions on test data.

This function will be called on every `moon` of the `Out-Of-Sample`.

In [ ]:
# Uncomment what you need!
def infer(
    X_test: pd.DataFrame,
    model_directory_path: str,
    # loop_moon: int,
    # embargo: int,
) -> pd.DataFrame:
    """
    Do your inference here.
    This function will load the model(s) saved at the previous iteration and use it/them to produce your inference on the current moon.
    It is mandatory to send your inferences with the ids and moon so the system can match it correctly.

    Args:
        X_test: the independant variables of the current moon passed to your model.
        model_directory_path: the path to the directory to the directory in wich we will be saving your updated model.
        loop_moon: the moon currently being processed
        embargo: data embargo

    Returns:
        A dataframe (moon, id, prediction) with the inferences of your model for the current moon.
    """

    # Creating the predicted label dataframe with correct moons and ids
    prediction = X_test[["id", "moon"]].copy()

    # Loading the model saved by the train function
    model_path = get_model_path(model_directory_path)
    model = joblib.load(model_path)

    feature_columns = get_feature_columns(X_test)
    prediction["prediction"] = model.predict(X_test[feature_columns])

    return prediction

## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [ ]:
# Uncomment to clear up a bit of RAM by unloading some data

# import gc

# X_train.drop(X_train.index, inplace=True)
# del X_train

# y_train.drop(y_train.index, inplace=True)
# del y_train

# X_test.drop(X_test.index, inplace=True)
# del X_test

# gc.collect()

In [ ]:
crunch_tools.test(
    # Uncomment to disable the forced first train
    # force_first_train=False,
    force_first_train=True,

    # Uncomment to set the training frequency
    # train_frequency=2,  # train every 2 moons
    train_frequency=0,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)

## Results

Once the local tester is done, you can preview the result stored in `prediction/prediction.parquet`.

In [ ]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction

### Local scoring

You can call the function that the system uses to estimate your score locally.

A [Pearson correlation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) will be computed against the **targets**.

**Note**:
- If all predictions are constant, the correlation will be undefined. In this case, the score will be set to `0`.
- Predictions must be between `-1` and `1`.

In [ ]:
# Load the targets
y_test = pd.read_parquet(
    "data/y.reduced.parquet",
    filters=[
        ("moon", "in", prediction["moon"].unique())
    ]
)

y_test

In [ ]:
# Define the scoring function
def score(
    group: pd.DataFrame,
):
    prediction_column_name = f"prediction"
    target_column_name = f"target"

    return group[prediction_column_name].corr(
        group[target_column_name],
        method="pearson"
    )

# Merge the prediction with the targets with moon and id
merged = y_test.merge(
    prediction,
    on=["moon", "id"],
)

# Compute the pearson for each moon
pearson_values = merged\
    .groupby("moon")\
    .apply(score, include_groups=False)\
    .fillna(0)  # map constants to zero

pearson_values

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Create a run to validate it

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [ ]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# @markdown ---
# @markdown **Advanced:** Should the `requirements.txt` file be frozen using locally installed packages?
Pip_Freeze = True # @param {"type":"boolean"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
    include_installed_packages_version=Pip_Freeze,
)